In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import umap
from bokeh.io import output_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CDSView, GroupFilter
from bokeh.layouts import column
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10
import panel as pn
pn.extension()
%matplotlib inline

# 1) Dataset Selection and Processing

## Default Datasets:

There are three datasets this algorithm has built into the folder: 

1. Penguin attribute data
2. Coronary Artery Disease (CAD) and Valvular Heart Disease (VHD) Patient data
3. Breast Cancer Patient data

To select any of these datasets, comment out the appropriate code block ensuring that `directory`, `flag`, and `rowid` variables are correctly defined. It is worth mentioning that while the breast cancer data provides the most distinct clusters under UMAP transformation, it has the most data and requires a long EBM run-time. The only dataset that features multi state classification is the `flag = 'VHD'` version of the Coronary Artery Disease dataset.

## User Uploaded Datasets:

If a user is to upload a dataset, they must identify three things to run the UMAP-EBM tool: 

1. The path to the pandas `pd.read_csv()` compatible dataset
2. The (multi-state) classification column of interest labeled `flag` 
3. The name of the column in the dataset that functions as a row id. If no such column exists, leave `rowid = None`






In [30]:
# ### Penguins
# directory = "penguins.csv"
# flag = 'species'
# rowid = 'rowid'


# ### Coronary Artery Disease
# directory = "CAD.csv"
# # flag = 'VHD'        # This is a multi state classification field reflecting the 4 stages of Valvular Heart Disease.
# flag = 'Cath'
# rowid = None


### Breast Cancer
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
dat = data.data
y = data.target
feature_names = data.feature_names
df_breast_cancer = pd.DataFrame(data=dat, columns=feature_names)
df_breast_cancer['flag'] = y
df_breast_cancer
df = df_breast_cancer
flag = 'flag'
rowid = None

### Data Cleaning and Re-formatting

The algorithm is only compatible with numeric data and one `flag` column. At this stage we also remove any `rowid` columns that may be hard coded into an inputed csv... as is with the penguins data. If this column is left, it will be considered as a field and cause problems in both the UMAP and EBM steps.

In [ ]:
# df = pd.read_csv(directory)       # Comment this out when running the Breast Cancer Data

color_classifier = df[flag]

df = df.dropna()
df[flag].value_counts()
if rowid is not None:
    df = df.drop(rowid, axis=1)

df_numeric = df.select_dtypes(include=["int64","float64"])

scaled_df_numeric = StandardScaler().fit_transform(df_numeric.values) # Mean centers and standardizes data:
df_numeric


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,flag
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


# 2) UMAP Dimension Reduction and Bokeh Plotting
### With Bokeh's lasso selection tool

Experimenting heavily with UMAP was not the scope of this project. As such, we use the default parameters of the python UMAP library.

In [ ]:

# The ENTIRE UMAP algorithm in 2 lines

reducer = umap.UMAP()       # I chose not to alter parameters
embedding = reducer.fit_transform(scaled_df_numeric)



# BOKEH Interactive Lasso Plotting

# 1) Create a master source with everything
data = {
'x':         embedding[:,0],
'y':         embedding[:,1],
'size':      [8]*len(embedding),
flag:   [str(s) for s in color_classifier], 
'rowid':     list(range(len(embedding)))
}
source = ColumnDataSource(data)


# 2) Build factors and palette for color coding.
factors = sorted(set(data[flag]))
n = len(factors)
key = min(max(n, 3), 10)        # Color codings have to be at least 3 for bokeh, and we want no more than 10 for ease on the eyes
base_palette = Category10[key]  
palette = base_palette[:n]  

# 3) Plot the source incorporating Bokeh's lasso feature.
p = figure(tools="lasso_select", width=600, height=600)

for sp, color in zip(factors, palette):
    view = CDSView(filter=GroupFilter(column_name=flag, group=sp))
    p.circle(
        'x', 'y',
        source=source,
        view=view,
        size='size',
        fill_color=color,
        line_color=None,
        legend_label=sp
    )

p.legend.title        = flag
p.legend.location     = "top_right"
p.legend.click_policy = "hide"
p.legend.background_fill_alpha = 0.8

# 4) Incorporate "onclick" interactive feature.
output = pn.pane.Str("")
selected_indices = []

def save_indices(event=None):
    global selected_indices
    # now these are global indices into `source.data`
    selected_indices = source.selected.indices
    output.object = f"Saved {len(selected_indices)} indices: {selected_indices}"

button = pn.widgets.Button(name="Save Selection")
button.on_click(save_indices)

pn.Column(p, button, output)


/var/folders/qp/47sbgf9n72n0thwvv8hlq7jw0000gn/T/ipykernel_33400/3980080484.py:34: BokehDeprecationWarning:

'circle() method with size value' was deprecated in Bokeh 3.4.0 and will be removed, use 'scatter(size=...) instead' instead.

/var/folders/qp/47sbgf9n72n0thwvv8hlq7jw0000gn/T/ipykernel_33400/3980080484.py:34: BokehDeprecationWarning:

'circle() method with size value' was deprecated in Bokeh 3.4.0 and will be removed, use 'scatter(size=...) instead' instead.



BokehModel(combine_events=True, render_bundle={'docs_json': {'883c192f-f9f4-432c-9a7a-3478584d7bf0': {'version…

# 3) EBM Interpretability Extraction 

## <span style="color:red;">WARNING:</span>

Ensure to select a cluster from the Bokeh plot's UMAP transformation before running the EBM code block. <span style="color:red;">The session may be ruined if this precaution is not made</span>. See trouble shooting suggestions if this occurs...

### Trouble Shooting:

- Restart the kernel 
- Delete and reopen the auto UMAP EBM tab from the IDE
- Copy all code cells into a new notebook 
- All of the above

## Running the EBM:

When a selection is made, uncomment the code below and run the cell. The code has worked if show(ebm.explain_global()) outputs an orange, sideways bar plot.

In [ ]:
# from interpret.glassbox import ExplainableBoostingClassifier
# from interpret import show

# from interpret import set_visualize_provider
# from interpret.provider import InlineProvider
# set_visualize_provider(InlineProvider())

# data = df.drop(columns=flag)

# X = pd.DataFrame(data=data)
# y = np.where(X.index.isin(selected_indices), 0, 1)


# # Split the data into training and test sets
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2
# )

# # Initialize and train the Explainable Boosting Machine (EBM)
# ebm = ExplainableBoostingClassifier()
# ebm.fit(X_train, y_train)

# # Make predictions and evaluate the model
# y_pred = ebm.predict(X_test)
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))      # summary statistcis for the user to assess confidence of explainability.

# show(ebm.explain_global())

Accuracy: 0.9736842105263158

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.96      0.97        49
           1       0.97      0.98      0.98        65

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



# Conclusion

This notebook can be used to identify key data matrix features that influence the clustering of data in UMAP dimension reduced space. The EBM serves to explain the mapping patterns of the seemingly random and ambiguous UMAP algorithm.